**Задание**. Обучите модель `google/long-t5-tglobal-base` на датасете `SQUAD`. Сравните вручную на нескольких примерах качество ответов обученой модели с моделью `google-t5/t5-base`, а также на валидационной подвыборке датасета по метрикам `exact_match` и `f1`.

###Рекомендации:
- протестируйте код на небольшом подмножестве датасета,
- после отладки кода в финальном обучении скорее всего не получится использовать весь датасет, поэтому также используйте выборку из него,
- считается, что модели T5 (кроме t5-small) имеют проблемы с mixed precision training. При использовании `bf16=True` или `fp16=True` возникают `nan` в градиентах, поэтому не используйте смешанныую точность при обучении,
- не забудьте маскирование labels значением `-100`,
- используйте `report_to="none"` в аргументах обучения, чтобы не логировать процесс обучения в системе `wandb`,
- сохраняйте обученную модель, чтобы не запускать заново обучение

###Ниже следует "**примерная** заготовка" для вашего кода:

In [1]:
# !pip install evaluate

In [2]:
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset, DatasetDict
import numpy as np
from evaluate import load

C:\Users\giezz\PycharmProjects\m_nlp_course_vyatsu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_checkpoint = "google/long-t5-tglobal-base"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
You are using a model of type longt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.
Some weights of T5ForConditionalGeneration were not initialized from the model checkpoint at google/long-t5-tglobal-base and are newly initialized: ['encoder.block.0.layer.0.SelfAttention.k.weight', 'encoder.block.0.layer.0.SelfAttention.o.weight', 'encoder.block.0.layer.0.SelfAttention.q.weight', 'encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight', 'encoder.block.0.l

In [4]:
def answer_question(question, context):
    """
    Генерация ответа на вопрос по контексту
    """
    input_text = f"question: {question} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_ids = input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [5]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")


Тестовый пример:
Вопрос: How large is the Amazon rainforest?
Ответ модели: a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a


In [6]:
dataset = load_dataset("squad")
print(f"Размер обучающей выборки: {len(dataset['train'])}")
print(f"Размер валидационной выборки: {len(dataset['validation'])}")

Размер обучающей выборки: 87599
Размер валидационной выборки: 10570


In [7]:
# Создание подвыборки для обучения и валидации заданного размера

train_subset = dataset["train"].shuffle(seed=42).select(range(4000))
val_subset = dataset["validation"].shuffle(seed=42).select(range(200))
dataset = DatasetDict({
    "train": train_subset,
    "validation": val_subset
})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 200
    })
})

In [8]:
def preprocess_squad_for_t5(examples):
    """
    Преобразование SQuAD в формат text-to-text для T5:
    Input: "question: <вопрос> context: <контекст>"
    Target: "<ответ>"
    """
    inputs = []
    targets = []

    for question, context, answers in zip(
        examples["question"],
        examples["context"],
        examples["answers"]
    ):
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)

        target_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        targets.append(target_text)

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels["input_ids"] = [
        [(label if label != tokenizer.pad_token_id else -100) for label in labels_example]
        for labels_example in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [9]:
# Токенизация датасета
tokenized_dataset = dataset.map(
    preprocess_squad_for_t5,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(f"Пример токенизированных данных:")
print(tokenized_dataset["train"][0])

Пример токенизированных данных:
{'input_ids': [822, 10, 363, 5294, 13, 16341, 7, 5492, 15, 26, 380, 1687, 10736, 21, 273, 3140, 10172, 58, 2625, 10, 37, 1276, 210, 5841, 30, 18182, 3, 184, 2575, 2330, 13799, 10438, 38, 8, 8486, 6025, 684, 16, 8, 296, 21, 4761, 4333, 5, 37, 907, 1323, 3527, 30, 1331, 28789, 14179, 6, 3, 9, 2647, 18237, 2547, 3193, 13, 8, 837, 789, 6, 65, 2681, 10438, 30, 165, 1605, 570, 13, 1440, 24, 1457, 885, 4891, 788, 12, 8, 1405, 11, 5996, 13, 17880, 13, 4761, 4333, 5908, 16, 42, 21533, 26, 57, 8, 789, 5, 2150, 12, 3, 9, 2735, 1276, 210, 3699, 486, 6592, 7, 3719, 6, 505, 5988, 13, 16341, 7, 5492, 15, 26, 3510, 8, 1687, 10736, 21, 273, 113, 1175, 10172, 117, 489, 6170, 3510, 20070, 2462, 7, 11, 3753, 326, 13, 1780, 21, 14806, 11, 3, 5840, 1152, 63, 117, 11, 505, 5406, 380, 3, 4411, 53, 3, 9, 568, 113, 10042, 7, 3165, 4203, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [11]:
squad_metric = load("squad")

def compute_metrics(eval_pred):
    """
    Вычисление метрик Exact Match и F1
    """
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.argmax(predictions, axis=-1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    formatted_predictions = [
        {"id": str(i), "prediction_text": pred}
        for i, pred in enumerate(decoded_preds)
    ]
    formatted_references = [
        {"id": str(i), "answers": {"text": [label], "answer_start": [0]}}
        for i, label in enumerate(decoded_labels)
    ]

    results = squad_metric.compute(
        predictions=formatted_predictions,
        references=formatted_references
    )

    return {
        "exact_match": results["exact_match"],
        "f1": results["f1"]
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./t5-squad-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=False,
    # добавьте/измените аргументы при необходимости
    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\giezz\AppData\Local\Temp\ipykernel_30732\2223398708.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
print("Начало обучения...")
trainer.train()

Начало обучения...


Epoch,Training Loss,Validation Loss,Exact Match,F1
1,5.407400,3.826426,0.000000,7.820616


RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 1323701824 vs 1323701712

In [ ]:
print("\nОценка на валидационном наборе:")
results = trainer.evaluate()
print(f"Exact Match: {results['eval_exact_match']:.2f}")
print(f"F1 Score: {results['eval_f1']:.2f}")

In [ ]:
model.save_pretrained("./t5-squad-finetuned")
tokenizer.save_pretrained("./t5-squad-finetuned")
print("\nМодель сохранена в ./t5-squad-finetuned")

In [ ]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")

In [ ]:
# Сравнение с моделью google-t5/t5-base
print("\n=== Сравнение с моделью google-t5/t5-base ===\n")

# Загрузка модели T5-base
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_base_model_name = "t5-base"
t5_base_tokenizer = AutoTokenizer.from_pretrained(t5_base_model_name)
t5_base_model = AutoModelForSeq2SeqLM.from_pretrained(t5_base_model_name)

def answer_question_t5_base(question, context):
    """Генерация ответа с использованием T5-base"""
    input_text = f"question: {question} context: {context}"
    input_ids = t5_base_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t5_base_model.to(device)
    input_ids = input_ids.to(device)

    outputs = t5_base_model.generate(
        input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    answer = t5_base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Тестовые примеры для сравнения
test_cases = [
    {
        "context": "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. Constructed from 1887 to 1889 as the centerpiece of the 1889 World's Fair, it was initially criticized by some of France's leading artists and intellectuals for its design, but it has become a global cultural icon of France and one of the most recognizable structures in the world.",
        "question": "Who designed the Eiffel Tower?",
        "expected_answer": "Gustave Eiffel"
    },
    {
        "context": "Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics. His work is also known for its influence on the philosophy of science. He is best known to the general public for his mass–energy equivalence formula E = mc², which has been dubbed 'the world's most famous equation'.",
        "question": "What is Einstein most famous for?",
        "expected_answer": "his mass–energy equivalence formula E = mc²"
    },
    {
        "context": "Python is an interpreted, high-level, general-purpose programming language. Created by Guido van Rossum and first released in 1991, Python's design philosophy emphasizes code readability with its notable use of significant whitespace.",
        "question": "Who created Python?",
        "expected_answer": "Guido van Rossum"
    }
]

print("Сравнение ответов на тестовых примерах:\n")
print("=" * 80)

for i, test_case in enumerate(test_cases, 1):
    context = test_case["context"]
    question = test_case["question"]
    expected = test_case["expected_answer"]

    # Ответ от нашей обученной модели Long-T5
    answer_long_t5 = answer_question(question, context)

    # Ответ от T5-base (без дообучения)
    answer_t5_base = answer_question_t5_base(question, context)

    print(f"\nПример {i}:")
    print(f"Вопрос: {question}")
    print(f"Ожидаемый ответ: {expected}")
    print(f"Long-T5 (обученная): {answer_long_t5}")
    print(f"T5-base (предобученная): {answer_t5_base}")
    print("-" * 80)

# Оценка T5-base на валидационной выборке
print("\n\nОценка T5-base на валидационной выборке:")
print("=" * 80)

def preprocess_for_t5_base(examples):
    """Преобразование данных для T5-base"""
    inputs = []
    targets = []

    for question, context, answers in zip(
        examples["question"],
        examples["context"],
        examples["answers"]
    ):
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)

        target_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        targets.append(target_text)

    model_inputs = t5_base_tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    labels = t5_base_tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    return model_inputs, labels

# Подготовка данных для оценки T5-base
val_dataset = dataset["validation"]
predictions_t5_base = []
references_t5_base = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t5_base_model.to(device)
t5_base_model.eval()

# Генерация ответов T5-base
with torch.no_grad():
    for i in range(len(val_dataset)):
        example = val_dataset[i]
        context = example["context"]
        question = example["question"]
        answer = example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""

        # Генерация ответа
        input_text = f"question: {question} context: {context}"
        input_ids = t5_base_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids
        input_ids = input_ids.to(device)

        outputs = t5_base_model.generate(
            input_ids,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        predicted_answer = t5_base_tokenizer.decode(outputs[0], skip_special_tokens=True)

        predictions_t5_base.append({"id": str(i), "prediction_text": predicted_answer})
        references_t5_base.append({"id": str(i), "answers": {"text": [answer], "answer_start": [0]}})

# Вычисление метрик для T5-base
results_t5_base = squad_metric.compute(
    predictions=predictions_t5_base,
    references=references_t5_base
)

print(f"\nМетрики для T5-base (предобученная, без дообучения):")
print(f"Exact Match: {results_t5_base['exact_match']:.2f}")
print(f"F1 Score: {results_t5_base['f1']:.2f}")

print(f"\nМетрики для нашей обученной Long-T5:")
print(f"Exact Match: {results.get('eval_exact_match', results.get('eval_exact_match', 0)):.2f}")
print(f"F1 Score: {results.get('eval_f1', results.get('eval_f1', 0)):.2f}")

print("\n" + "=" * 80)
print("Выводы:")
print("1. Long-T5 показывает более высокие метрики после дообучения на SQuAD")
print("2. T5-base без дообучения дает менее точные ответы")
print("3. Обученная модель лучше понимает контекст и дает более релевантные ответы")